
# [Step 1] 라이브러리 설치, 폴더 세팅 및 100종목 데이터 수집


In [ ]:
# 1. 필수 라이브러리 설치 및 임포트

!pip install yfinance pandas numpy -q

import os
import time
from pathlib import Path
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta

In [ ]:
# 2. 프로젝트 폴더 세팅 (코랩 로컬 디렉토리 구조)
from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = Path("/content/drive/MyDrive/datamining_project")
DATA_RAW = BASE_DIR / "data/raw"
DATA_PROCESSED = BASE_DIR / "data/processed"
RESULTS_DIR = BASE_DIR / "results"
FIGURES_DIR = RESULTS_DIR / "figures"

for p in [DATA_RAW, DATA_PROCESSED, RESULTS_DIR, FIGURES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("[+] 코랩 데이터 디렉토리 세팅 완료!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[+] 코랩 데이터 디렉토리 세팅 완료!


In [ ]:
# 3. 수집 기간 정의 (어제 기준 2년 6개월 역산)
end_date = datetime.today() - timedelta(days=1)
start_date = end_date - timedelta(days=30 * 30)
start_str = start_date.strftime("%Y-%m-%d")
end_str = end_date.strftime("%Y-%m-%d")

In [ ]:
# 4. 대형주(35), 중형주(35), 소형주(30) 종목 정의
# KRX(한국거래소) 공식 시가총액 기준

# 대형주 : 코스피/코스닥 시장에서 시가 총액 1위부터 100까지의 기업
# 중형주 : 코스피/코스닥 시장에서 시가 총액 101위부터 300위까지의 기업
# 소형주 : 코스피/코스닥 시장에서 시가 총액 301위 이하 기업

large_caps = [
    "005930.KS", "000660.KS", "073540.KS", "207940.KS", "005490.KS",
    "005380.KS", "035420.KS", "051910.KS", "035720.KS", "006400.KS",
    "012330.KS", "068270.KS", "105560.KS", "055550.KS", "015760.KS",
    "000270.KS", "017670.KS", "096770.KS", "032830.KS", "003550.KS",
    "033780.KS", "000810.KS", "018260.KS", "086790.KS", "010950.KS",
    "009150.KS", "051900.KS", "034730.KS", "251270.KS", "011170.KS",
    "009830.KS", "034220.KS", "010130.KS", "004020.KS", "016360.KS"
]
mid_caps = [
    "042660.KS", "078930.KS", "000720.KS", "028260.KS", "036570.KS",
    "009240.KS", "008930.KS", "000100.KS", "010620.KS", "069960.KS",
    "001040.KS", "004990.KS", "021240.KS", "005940.KS", "006260.KS",
    "192820.KS", "161390.KS", "285130.KS", "023530.KS", "007070.KS",
    "047040.KS", "001450.KS", "000120.KS", "079160.KS", "003240.KS",
    "064350.KS", "004370.KS", "052690.KS", "011210.KS", "020150.KS",
    "001800.KS", "000080.KS", "005830.KS", "006360.KS", "003490.KS"
]
small_caps = [
    "083660.KQ", "044180.KQ", "011040.KQ", "021040.KQ", "054340.KQ",
    "053030.KQ", "032960.KQ", "038340.KQ", "037350.KQ", "115530.KQ",
    "023160.KQ", "024840.KQ", "036480.KQ", "039560.KQ", "058450.KQ",
    "065560.KQ", "101330.KQ", "054050.KQ", "036640.KQ", "041510.KQ",
    "025950.KQ", "060230.KQ", "060250.KQ", "035620.KQ", "024810.KQ",
    "038110.KQ", "043260.KQ", "065620.KQ", "052420.KQ", "033600.KQ"
]

all_tickers = large_caps + mid_caps + small_caps
ticker_group_map = {t: "Large" for t in large_caps}
ticker_group_map.update({t: "Medium" for t in mid_caps})
ticker_group_map.update({t: "Small" for t in small_caps})

In [ ]:
# 5. 루프를 돌며 종목별 데이터 다운로드 및 병합
all_frames = []
drop_count = 0

for idx, ticker in enumerate(all_tickers, start=1):
    group = ticker_group_map[ticker]
    print(f"    [{idx}/{len(all_tickers)}] 다운로드 중: {ticker} ({group})")

    try:
        # yfinance 다운로드 (수급 데이터가 없으므로 프록시나 차단 우려가 적어 깔끔하게 가져옵니다)
        df = yf.download(ticker, start=start_str, end=end_str, progress=False, auto_adjust=False)

        # 데이터가 비어있거나 정상적이지 않은 경우 패스
        if df.empty or len(df) < 10:
            print(f"    [!] {ticker} 데이터가 부족하거나 존재하지 않아 제외합니다.")
            drop_count += 1
            continue

        # 멀티인덱스 컬럼 구조 분해 (yfinance 최신 버전 대응)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df = df.reset_index()

        # 컬럼명을 소문자로 통일하여 데이터마이닝 전처리 편의성 확보
        df = df.rename(columns={
            "Date": "date", "Open": "open", "High": "high",
            "Low": "low", "Close": "close", "Adj Close": "adj_close",
            "Volume": "volume"
        })

        # 종목 메타 정보 삽입
        df["ticker"] = ticker
        df["group"] = group

        all_frames.append(df)

        # Yahoo API 과부하 방지 및 안정적인 수집을 위한 미세 타임슬립
        time.sleep(0.1)

    except Exception as e:
        print(f"    [-] {ticker} 수집 중 에러 발생: {e}")
        drop_count += 1

    [1/100] 다운로드 중: 005930.KS (Large)
    [2/100] 다운로드 중: 000660.KS (Large)
    [3/100] 다운로드 중: 073540.KS (Large)
    [4/100] 다운로드 중: 207940.KS (Large)
    [5/100] 다운로드 중: 005490.KS (Large)
    [6/100] 다운로드 중: 005380.KS (Large)
    [7/100] 다운로드 중: 035420.KS (Large)
    [8/100] 다운로드 중: 051910.KS (Large)
    [9/100] 다운로드 중: 035720.KS (Large)
    [10/100] 다운로드 중: 006400.KS (Large)
    [11/100] 다운로드 중: 012330.KS (Large)
    [12/100] 다운로드 중: 068270.KS (Large)
    [13/100] 다운로드 중: 105560.KS (Large)
    [14/100] 다운로드 중: 055550.KS (Large)
    [15/100] 다운로드 중: 015760.KS (Large)
    [16/100] 다운로드 중: 000270.KS (Large)
    [17/100] 다운로드 중: 017670.KS (Large)
    [18/100] 다운로드 중: 096770.KS (Large)
    [19/100] 다운로드 중: 032830.KS (Large)
    [20/100] 다운로드 중: 003550.KS (Large)
    [21/100] 다운로드 중: 033780.KS (Large)
    [22/100] 다운로드 중: 000810.KS (Large)
    [23/100] 다운로드 중: 018260.KS (Large)
    [24/100] 다운로드 중: 086790.KS (Large)
    [25/100] 다운로드 중: 010950.KS (Large)
    [26/100] 다운로드 중: 009150.KS (La

ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['010620.KS']: YFTzMissingError('possibly delisted; no timezone found')


    [!] 010620.KS 데이터가 부족하거나 존재하지 않아 제외합니다.
    [45/100] 다운로드 중: 069960.KS (Medium)
    [46/100] 다운로드 중: 001040.KS (Medium)
    [47/100] 다운로드 중: 004990.KS (Medium)
    [48/100] 다운로드 중: 021240.KS (Medium)
    [49/100] 다운로드 중: 005940.KS (Medium)
    [50/100] 다운로드 중: 006260.KS (Medium)
    [51/100] 다운로드 중: 192820.KS (Medium)
    [52/100] 다운로드 중: 161390.KS (Medium)
    [53/100] 다운로드 중: 285130.KS (Medium)
    [54/100] 다운로드 중: 023530.KS (Medium)
    [55/100] 다운로드 중: 007070.KS (Medium)
    [56/100] 다운로드 중: 047040.KS (Medium)
    [57/100] 다운로드 중: 001450.KS (Medium)
    [58/100] 다운로드 중: 000120.KS (Medium)
    [59/100] 다운로드 중: 079160.KS (Medium)
    [60/100] 다운로드 중: 003240.KS (Medium)
    [61/100] 다운로드 중: 064350.KS (Medium)
    [62/100] 다운로드 중: 004370.KS (Medium)
    [63/100] 다운로드 중: 052690.KS (Medium)
    [64/100] 다운로드 중: 011210.KS (Medium)
    [65/100] 다운로드 중: 020150.KS (Medium)
    [66/100] 다운로드 중: 001800.KS (Medium)
    [67/100] 다운로드 중: 000080.KS (Medium)
    [68/100] 다운로드 중: 005830.KS (Medi

ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['054340.KQ']: YFTzMissingError('possibly delisted; no timezone found')


    [!] 054340.KQ 데이터가 부족하거나 존재하지 않아 제외합니다.
    [76/100] 다운로드 중: 053030.KQ (Small)
    [77/100] 다운로드 중: 032960.KQ (Small)


...

    [!] 024810.KQ 데이터가 부족하거나 존재하지 않아 제외합니다.
    [96/100] 다운로드 중: 038110.KQ (Small)
    [97/100] 다운로드 중: 043260.KQ (Small)
    [98/100] 다운로드 중: 065620.KQ (Small)


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['065620.KQ']: YFTzMissingError('possibly delisted; no timezone found')


    [!] 065620.KQ 데이터가 부족하거나 존재하지 않아 제외합니다.
    [99/100] 다운로드 중: 052420.KQ (Small)


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['033600.KQ']: YFTzMissingError('possibly delisted; no timezone found')


    [100/100] 다운로드 중: 033600.KQ (Small)
    [!] 033600.KQ 데이터가 부족하거나 존재하지 않아 제외합니다.


In [ ]:
# 6. 전체 수집 데이터 하나의 데이터프레임으로 통합
# 최초 시장규모별 지수 기준에 따라 100개의 후보 종목 선정

# README.md 파일에 밑에 제가 쓴 부분은 꼭 넣어주세요
# 최초 시장규모별 100개의 후보 종목을 선정하였으나, 데이터 연속성 확보 및
# 신규 상장주, 거래정지 종목 제외 등 데이터 정제 과정을 거쳐 최종적으로 93개의
# 종목을 분석 대상으로 확정하였습니다.

if all_frames:
    raw_total_df = pd.concat(all_frames, ignore_index=True)
    raw_total_df["date"] = pd.to_datetime(raw_total_df["date"])
    raw_total_df = raw_total_df.sort_values(["ticker", "date"]).reset_index(drop=True)

    # 5. 지정하신 DATA_RAW 경로에 csv 파일로 보존
    raw_file_path = DATA_RAW / "raw_ohlcv_100.csv"
    raw_total_df.to_csv(raw_file_path, index=False, encoding="utf-8-sig")

    print("\n" + "="*50)
    print(f"[+] 수집 완료 및 파일 저장 성공!")
    print(f"[+] 최종 저장 경로: {raw_file_path.resolve()}")
    print(f"[+] 수집된 총 행(Instance) 수: {len(raw_total_df)}개")
    print(f"[+] 유실/상장폐지 의심 제외 종목 수: {drop_count}개")
    print("="*50)
else:
    print("[-] 수집된 데이터가 전혀 없습니다. 티커나 네트워크 상태를 확인하세요.")


[+] 수집 완료 및 파일 저장 성공!
[+] 최종 저장 경로: /content/drive/MyDrive/datamining_project/data/raw/raw_ohlcv_100.csv
[+] 수집된 총 행(Instance) 수: 55428개
[+] 유실/상장폐지 의심 제외 종목 수: 7개
